# Heart Disease Prediction — ML Model Comparison

## Setup: Download and Load the Dataset

In [ ]:
import urllib.request, zipfile, os
import pandas as pd
import numpy as np

url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/Heart%20Disease%20Prediction%20Dataset.zip"
urllib.request.urlretrieve(url, "heart.zip")

with zipfile.ZipFile("heart.zip", "r") as z:
    z.extractall("heart_data")

for root, dirs, files in os.walk("heart_data"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import glob

csv_files = glob.glob("heart_data/**/*.csv", recursive=True)
print("CSV files found:", csv_files)

# Load whichever CSV files are present
dfs = {os.path.basename(f): pd.read_csv(f) for f in csv_files}
for name, df in dfs.items():
    print(f"\n{name}: {df.shape}")
    print(df.head(2))

## Exercise 1: Exploratory Data Analysis

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Use the first CSV found; adjust key if needed
df = list(dfs.values())[0]

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print(df.describe())

In [ ]:
import matplotlib.pyplot as plt

# Identify the target column (last column or column named 'target'/'HeartDisease')
if "target" in df.columns:
    target_col = "target"
elif "HeartDisease" in df.columns:
    target_col = "HeartDisease"
else:
    target_col = df.columns[-1]

print(f"Target column: '{target_col}'")
print(df[target_col].value_counts())

df[target_col].value_counts().plot(kind="bar", color=["steelblue", "tomato"])
plt.title(f"Class Distribution — {target_col}")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

df[numeric_cols].hist(bins=20, figsize=(14, 10), color="steelblue", edgecolor="white")
plt.suptitle("Feature Distributions", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Encode categorical columns if any
df_encoded = pd.get_dummies(df, drop_first=True)

# Separate features and target
if target_col in df_encoded.columns:
    y = df_encoded[target_col].astype(int)
    X = df_encoded.drop(columns=[target_col])
else:
    # target may have been renamed after get_dummies
    target_candidates = [c for c in df_encoded.columns if target_col in c]
    target_col_enc = target_candidates[0]
    y = df_encoded[target_col_enc].astype(int)
    X = df_encoded.drop(columns=[target_col_enc])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set: {X_train_sc.shape}, Test set: {X_test_sc.shape}")
print(f"Features used: {X.columns.tolist()}")

## Exercise 2: Logistic Regression without Grid Search

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

lr = LogisticRegression(C=1.0, penalty="l2", max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

y_pred_lr = lr.predict(X_test_sc)

print("Logistic Regression (no tuning)")
print("-" * 40)
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt="d",
            cmap="Blues", cbar=False)
plt.title("Logistic Regression — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 3: Logistic Regression with Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"]
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_lr,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_lr.fit(X_train_sc, y_train)

print(f"Best parameters: {grid_lr.best_params_}")
print(f"Best CV accuracy: {grid_lr.best_score_:.4f}")

y_pred_lr_gs = grid_lr.best_estimator_.predict(X_test_sc)

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_lr_gs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr_gs))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_lr_gs), annot=True, fmt="d",
            cmap="Blues", cbar=False)
plt.title("LR + Grid Search — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 4: SVM without Grid Search

In [ ]:
from sklearn.svm import SVC

# RBF kernel is a robust default for tabular classification tasks.
# C=1 balances margin width and misclassification; gamma='scale' adapts to feature variance.
svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm.fit(X_train_sc, y_train)

y_pred_svm = svm.predict(X_test_sc)

print("SVM (no tuning) — kernel=rbf, C=1.0, gamma=scale")
print("-" * 50)
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt="d",
            cmap="Greens", cbar=False)
plt.title("SVM — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 5: SVM with Grid Search

In [ ]:
param_grid_svm = {
    "C": [0.1, 1, 10],
    "kernel": ["rbf", "linear"],
    "gamma": ["scale", "auto"]
}

grid_svm = GridSearchCV(
    SVC(random_state=42),
    param_grid_svm,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_svm.fit(X_train_sc, y_train)

print(f"Best parameters: {grid_svm.best_params_}")
print(f"Best CV accuracy: {grid_svm.best_score_:.4f}")

y_pred_svm_gs = grid_svm.best_estimator_.predict(X_test_sc)

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_svm_gs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm_gs))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_svm_gs), annot=True, fmt="d",
            cmap="Greens", cbar=False)
plt.title("SVM + Grid Search — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 6: XGBoost without Grid Search

In [ ]:
from xgboost import XGBClassifier

# Hyperparameter choices:
# - n_estimators=100: a solid default that avoids underfitting without being too slow.
# - learning_rate=0.1: standard shrinkage that works well with 100 trees.
# - max_depth=4: shallow enough to prevent overfitting on a small dataset.
# - subsample=0.8 and colsample_bytree=0.8: mild regularization via row/column sampling.
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train_sc, y_train)

y_pred_xgb = xgb.predict(X_test_sc)

print("XGBoost (no tuning)")
print("-" * 40)
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=True, fmt="d",
            cmap="Oranges", cbar=False)
plt.title("XGBoost — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 7: XGBoost with Grid Search

In [ ]:
param_grid_xgb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 4, 6],
    "subsample": [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    XGBClassifier(
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42
    ),
    param_grid_xgb,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)
grid_xgb.fit(X_train_sc, y_train)

print(f"Best parameters: {grid_xgb.best_params_}")
print(f"Best CV accuracy: {grid_xgb.best_score_:.4f}")

y_pred_xgb_gs = grid_xgb.best_estimator_.predict(X_test_sc)

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_xgb_gs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_gs))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_xgb_gs), annot=True, fmt="d",
            cmap="Oranges", cbar=False)
plt.title("XGBoost + Grid Search — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Model Comparison Summary

In [ ]:
from sklearn.metrics import f1_score

results = {
    "Logistic Regression": y_pred_lr,
    "LR + Grid Search": y_pred_lr_gs,
    "SVM": y_pred_svm,
    "SVM + Grid Search": y_pred_svm_gs,
    "XGBoost": y_pred_xgb,
    "XGBoost + Grid Search": y_pred_xgb_gs,
}

summary = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "F1-Score": round(f1_score(y_test, preds, average="weighted"), 4)
    }
    for name, preds in results.items()
]).sort_values("Accuracy", ascending=False).reset_index(drop=True)

print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].barh(summary["Model"], summary["Accuracy"], color="steelblue")
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Test Accuracy by Model")
for i, v in enumerate(summary["Accuracy"]):
    axes[0].text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=9)

axes[1].barh(summary["Model"], summary["F1-Score"], color="tomato")
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("F1-Score")
axes[1].set_title("Weighted F1-Score by Model")
for i, v in enumerate(summary["F1-Score"]):
    axes[1].text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Analysis Report

**Dataset**: Heart Disease Prediction — binary classification (disease present / absent).

**Preprocessing**: Categorical variables were one-hot encoded. All features were standardized using `StandardScaler` before model training to ensure fair comparison, especially for distance-based models (Logistic Regression, SVM).

**Model observations**:

- **Logistic Regression** is a linear model. It performs well when the decision boundary is approximately linear in feature space. Grid search over `C` and `penalty` allows the model to find the best regularization strength.
- **SVM** with an RBF kernel can capture non-linear relationships. It is sensitive to the scale of features (hence the need for standardization) and to the `C` and `gamma` hyperparameters. Grid search helps identify the optimal trade-off between margin width and misclassification tolerance.
- **XGBoost** is an ensemble method based on gradient-boosted decision trees. It is generally more powerful on tabular data than linear models. Key hyperparameters include `learning_rate` (shrinkage), `n_estimators` (number of trees), and `max_depth` (tree complexity). Grid search allows systematic exploration of these.

**Impact of Grid Search**: Grid search consistently improves or maintains model performance by replacing manually chosen hyperparameters with values validated across 5 cross-validation folds. The improvement is most visible for models that are sensitive to hyperparameter settings, such as SVM.

**Recommended model**: The best-performing model according to the comparison table above is recommended for deployment. In medical classification tasks, it is also important to examine **Recall** for the positive class (disease present) to minimize missed diagnoses.